# 04 - Latest HDF5 real vs CORSIKA peak matching

This notebook repeats the current peak-prioritized real-vs-sim comparison workflow, but replaces the old `10min.csv` input with only the latest Test_Panel HDF5 run:

- Panel A: `03A_040526_1h50m_AMP5_C5_thr0_82.h5`
- Panel B: `03B_040526_1h50m_AMP5_C5_thr0_123.h5`

Panels are loaded and segmented separately. Set `PRIMARY_PANEL = "A"` or `"B"` for the main comparison. Panels are not silently merged.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT / "src"))

DATA_CORSIKA = PROJECT_ROOT / "data/raw/corsika/DATA_PROCESADA_ANTONIO_ROMERO.csv"
LATEST_H5_FILES = {
    "A": PROJECT_ROOT / "data/raw/Test_Panel/03A_040526_1h50m_AMP5_C5_thr0_82.h5",
    "B": PROJECT_ROOT / "data/raw/Test_Panel/03B_040526_1h50m_AMP5_C5_thr0_123.h5",
}

OUTPUT_TABLES = PROJECT_ROOT / "outputs/tables"
OUTPUT_FIGURES = PROJECT_ROOT / "outputs/figures"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("CORSIKA exists:", DATA_CORSIKA.exists(), DATA_CORSIKA)
for panel, path in LATEST_H5_FILES.items():
    print(f"Latest HDF5 panel {panel} exists:", path.exists(), path)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from tambo_sipm import load_simulation_config
from tambo_sipm.io import (
    build_photon_events,
    detector_summary,
    estimate_sampling_interval_ns,
    format_test_panel_voltage_unit_report,
    load_romero_photon_table,
    load_test_panel_h5_samples,
    pulse_collection_to_dict,
    pulse_length_distribution,
    sampling_interval_report,
    segment_test_panel_samples,
    segment_test_panel_samples_by_threshold,
    summarize_test_panel_segmentation,
)
from tambo_sipm.analysis import (
    compare_real_sim_features,
    extract_features_from_pulse_table,
    summarize_feature_table,
)
from tambo_sipm.simulation import build_calibration_config
from tambo_sipm.simulation.batch import simulate_feature_row_from_photon_event
from tambo_sipm.detector import simulate_detector_event_from_config
from tambo_sipm.plotting import plot_waveform_overlay, save_figure

plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10
print("Imports OK")


## 1. Parameters

The Red Pitaya FPGA temporal resolution is 8 ns. Therefore every histogram that uses a temporal variable uses 8 ns bins. The estimated sampling interval is reported only as a consistency check.


In [ ]:
PRIMARY_PANEL = "A"  # "A" or "B"
assert PRIMARY_PANEL in {"A", "B"}

VOLTAGE_UNIT = "auto"  # "auto", "V", or "mV"
SEGMENTATION_METHOD = "gap"  # "gap" or "threshold"
GAP_THRESHOLD_NS = 50.0
MIN_SAMPLES_PER_PULSE = 2

THRESHOLD_SEGMENT_MV = 18.0
THRESHOLD_SEGMENT_PRE_SAMPLES = 3
THRESHOLD_SEGMENT_POST_SAMPLES = 3
THRESHOLD_SEGMENT_POLARITY = "positive"

FEATURE_THRESHOLD_MV = 18.0
N_PRETRIGGER_SAMPLES = 3
BASELINE_METHOD = "median"

RANDOM_SEED = 42
REPRESENTATIVE_REAL_PULSES = 1000
# Deadline-friendly mode: load only a leading window from each HDF5 file.
# Set to None for full-file processing after the delivery crunch.
H5_MAX_ROWS_PER_PANEL = 50_000
MAX_POOL_EVENTS = 20000
N_REPRESENTATIVE_OVERLAY = 60
N_REPRESENTATIVE_PAIR_PLOTS = 12

# Full A/B sample and pulse CSVs are very large. Keep these False for deadline runs;
# set True only when you explicitly need complete long tables on disk.
SAVE_FULL_PANEL_SAMPLE_TABLES = False
SAVE_FULL_PANEL_PULSE_TABLES = False

# Fixed methodology parameters; do not change unless explicitly requested.
VOLTAGE_SCALE_MV_PER_PE = 0.12
PDE = 0.41

# Current exploratory tuning values from the all-253 peak-matching notebook.
ETA_TRANSPORT = 0.025
TAU_F_NS = 220.0
ARRIVAL_SPREAD_NS = 20.0

print("Primary panel:", PRIMARY_PANEL)
print("Segmentation method:", SEGMENTATION_METHOD)
print("Voltage unit mode:", VOLTAGE_UNIT)


## 2. Load latest HDF5 samples separately by panel


In [ ]:
def panel_prefix(panel):
    return f"latest_h5_panel_{panel}"

samples_by_panel = {}
for panel, path in LATEST_H5_FILES.items():
    samples = load_test_panel_h5_samples(path, voltage_unit=VOLTAGE_UNIT, max_rows=H5_MAX_ROWS_PER_PANEL)
    samples_by_panel[panel] = samples
    print("=" * 80)
    print(f"Panel {panel}: {path.name}")
    print("Rows loaded:", len(samples))
    print("H5_MAX_ROWS_PER_PANEL:", H5_MAX_ROWS_PER_PANEL)
    display(samples[[
        "source_file", "panel", "run_id", "acquisition_date",
        "acquisition_duration", "threshold_token",
    ]].head(1))
    print(format_test_panel_voltage_unit_report(samples))
    print(sampling_interval_report(samples))
    display(samples[["sample_index", "adc_value", "voltage_raw", "voltage_mV", "time_total_seconds", "time_ns"]].head())
    display(samples[["adc_value", "voltage_raw", "voltage_mV", "time_ns"]].describe())
    if SAVE_FULL_PANEL_SAMPLE_TABLES:
        samples.to_csv(OUTPUT_TABLES / f"{panel_prefix(panel)}_samples.csv", index=False)
    else:
        samples.head(1000).to_csv(OUTPUT_TABLES / f"{panel_prefix(panel)}_samples_preview_1000.csv", index=False)


## 3. Segment pulses separately by panel

Gap segmentation is the default first attempt. Threshold segmentation is available as an explicit second option by setting `SEGMENTATION_METHOD = "threshold"` and exposing the threshold parameters above.


In [ ]:
def segment_panel_samples(samples):
    if SEGMENTATION_METHOD == "gap":
        return segment_test_panel_samples(samples, gap_threshold_ns=GAP_THRESHOLD_NS, min_samples=MIN_SAMPLES_PER_PULSE)
    if SEGMENTATION_METHOD == "threshold":
        return segment_test_panel_samples_by_threshold(
            samples,
            threshold_mV=THRESHOLD_SEGMENT_MV,
            pre_samples=THRESHOLD_SEGMENT_PRE_SAMPLES,
            post_samples=THRESHOLD_SEGMENT_POST_SAMPLES,
            min_samples=MIN_SAMPLES_PER_PULSE,
            polarity=THRESHOLD_SEGMENT_POLARITY,
        )
    raise ValueError("SEGMENTATION_METHOD must be 'gap' or 'threshold'.")

pulses_by_panel = {}
lengths_by_panel = {}
segmentation_rows = []

for panel, samples in samples_by_panel.items():
    pulses = segment_panel_samples(samples)
    lengths = pulse_length_distribution(pulses)
    summary = summarize_test_panel_segmentation(
        samples,
        pulses,
        feature_threshold_mV=FEATURE_THRESHOLD_MV,
        n_pretrigger_samples=N_PRETRIGGER_SAMPLES,
    )
    row = {
        "panel": panel,
        "source_file": samples["source_file"].iloc[0],
        "segmentation_method": SEGMENTATION_METHOD,
        "total_rows_loaded": summary["total_rows_loaded"],
        "estimated_sampling_interval_ns": summary["estimated_sampling_interval_ns"],
        "number_of_segmented_pulses": summary["number_of_segmented_pulses"],
        "voltage_min_mV": summary["voltage_range_mV"]["min"],
        "voltage_median_mV": summary["voltage_range_mV"]["median"],
        "voltage_max_mV": summary["voltage_range_mV"]["max"],
        "feature_threshold_mV": summary["feature_threshold_mV"],
        "pulses_passing_feature_threshold": summary["pulses_passing_feature_threshold"],
    }
    pulses_by_panel[panel] = pulses
    lengths_by_panel[panel] = lengths
    segmentation_rows.append(row)

    print("=" * 80)
    print(f"Panel {panel} segmentation summary")
    display(pd.DataFrame([row]))
    display(lengths.describe())
    display(lengths.head())
    if SAVE_FULL_PANEL_PULSE_TABLES:
        pulses.to_csv(OUTPUT_TABLES / f"{panel_prefix(panel)}_pulses.csv", index=False)
    lengths.to_csv(OUTPUT_TABLES / f"{panel_prefix(panel)}_pulse_lengths.csv", index=False)

segmentation_summary = pd.DataFrame(segmentation_rows)
segmentation_summary.to_csv(OUTPUT_TABLES / "latest_h5_segmentation_summary.csv", index=False)
display(segmentation_summary)


## 4. Extract features from HDF5 pulses


In [ ]:
def add_event_metadata(features, pulses):
    metadata_columns = [
        "event_id", "source_file", "panel", "run_id", "acquisition_date",
        "acquisition_duration", "threshold_token",
    ]
    metadata = pulses[metadata_columns].drop_duplicates("event_id")
    return features.merge(metadata, on="event_id", how="left")


def representative_event_ids_by_raw_peak(pulses, n_events, random_seed=42):
    """Pick a deterministic peak-spread sample of pulse event IDs."""
    pulse_peaks = (
        pulses.groupby("event_id", sort=True)
        .agg(
            raw_peak_mV=("voltage_mV", "max"),
            n_samples=("voltage_mV", "size"),
            duration_ns=("time_ns", lambda values: float(values.max() - values.min())),
        )
        .reset_index()
        .sort_values(["raw_peak_mV", "event_id"])
        .reset_index(drop=True)
    )

    if n_events is None or n_events >= len(pulse_peaks):
        selected = pulse_peaks.copy()
    else:
        positions = np.linspace(0, len(pulse_peaks) - 1, n_events).round().astype(int)
        selected = pulse_peaks.iloc[np.unique(positions)].copy()

        if len(selected) < n_events:
            missing = n_events - len(selected)
            remaining = pulse_peaks[~pulse_peaks["event_id"].isin(selected["event_id"])]
            selected = pd.concat(
                [selected, remaining.sample(n=missing, random_state=random_seed)],
                ignore_index=True,
            )

    return selected["event_id"].tolist(), selected


# The full segmented pulse tables for A/B are saved above. From here onward,
# use only a representative sample from PRIMARY_PANEL for feature extraction,
# matching, histograms, and overlays.
real_pulses_all_primary = pulses_by_panel[PRIMARY_PANEL]
selected_event_ids, representative_selection = representative_event_ids_by_raw_peak(
    real_pulses_all_primary,
    n_events=REPRESENTATIVE_REAL_PULSES,
    random_seed=RANDOM_SEED,
)

real_pulses = (
    real_pulses_all_primary[real_pulses_all_primary["event_id"].isin(selected_event_ids)]
    .copy()
    .reset_index(drop=True)
)

real_features = extract_features_from_pulse_table(
    real_pulses,
    threshold_mV=FEATURE_THRESHOLD_MV,
    n_pretrigger_samples=N_PRETRIGGER_SAMPLES,
    baseline_method=BASELINE_METHOD,
)
real_features = add_event_metadata(real_features, real_pulses)
real_waveforms = pulse_collection_to_dict(real_pulses)
real_sampling_interval_ns = estimate_sampling_interval_ns(samples_by_panel[PRIMARY_PANEL])

print("Selected panel:", PRIMARY_PANEL)
print("Total segmented pulses in selected panel:", real_pulses_all_primary["event_id"].nunique())
print("Representative real pulses selected:", real_pulses["event_id"].nunique())
print("Selected sampling interval [ns]:", real_sampling_interval_ns)
print("Feature rows:", len(real_features))
print("Valid:", int(real_features["valid"].sum()))
print("Invalid:", int((~real_features["valid"]).sum()))

print("Representative selection by raw peak:")
display(representative_selection[["event_id", "raw_peak_mV", "n_samples", "duration_ns"]].describe(include="all"))
display(representative_selection.head())

display(summarize_feature_table(real_features))
display(real_features[["event_id", "valid", "extraction_status", "raw_peak_mV", "peak_mV", "width_ns", "integral_mVns"]].head())
display(real_features[["raw_peak_mV", "peak_mV", "width_ns", "integral_mVns", "rms_mV"]].describe())

representative_selection.to_csv(
    OUTPUT_TABLES / f"{panel_prefix(PRIMARY_PANEL)}_representative_1000_selection.csv",
    index=False,
)
real_pulses.to_csv(
    OUTPUT_TABLES / f"{panel_prefix(PRIMARY_PANEL)}_representative_1000_pulses.csv",
    index=False,
)
real_features.to_csv(
    OUTPUT_TABLES / f"{panel_prefix(PRIMARY_PANEL)}_representative_1000_features.csv",
    index=False,
)


In [ ]:
corsika_table = load_romero_photon_table(DATA_CORSIKA)
print("Detector summary:")
display(detector_summary(corsika_table))

available_detectors = sorted(corsika_table["Detector"].dropna().astype(int).unique())
print("Available detectors:", available_detectors)
print("Exploratory note: using all CORSIKA detectors for the simulated pool.")

def build_all_detector_events(corsika_dataframe):
    event_tables = []
    for detector in available_detectors:
        events = build_photon_events(corsika_dataframe, detector=int(detector)).copy()
        events["detector"] = int(detector)
        if "source_row" in events.columns:
            events["event_id"] = [f"corsika_d{int(detector)}_row{int(row)}" for row in events["source_row"]]
        else:
            events["event_id"] = [f"corsika_d{int(detector)}_idx{idx}" for idx in range(len(events))]
        event_tables.append(events)
    return pd.concat(event_tables, ignore_index=True)

corsika_events_all = build_all_detector_events(corsika_table)
print("CORSIKA events using all detectors:", len(corsika_events_all))
display(corsika_events_all.head())
display(corsika_events_all["detector"].value_counts().sort_index())
corsika_events_all.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_corsika_all_detectors_events.csv", index=False)


## 6. Detector simulation configuration


In [ ]:
base_config = load_simulation_config()
matched_config = build_calibration_config(
    base_config,
    transport_efficiency=ETA_TRANSPORT,
    tau_f_ns=TAU_F_NS,
    arrival_spread_ns=ARRIVAL_SPREAD_NS,
)
assert matched_config["voltage_conversion"]["voltage_scale_mV_per_pe"] == VOLTAGE_SCALE_MV_PER_PE
assert matched_config["photon_transport"]["pde"] == PDE

print("Fixed voltage_scale_mV_per_pe:", matched_config["voltage_conversion"]["voltage_scale_mV_per_pe"])
print("Fixed PDE:", matched_config["photon_transport"]["pde"])
print("transport_efficiency:", matched_config["photon_transport"]["transport_efficiency"])
print("tau_f_ns:", matched_config["sipm_response"]["tau_f_ns"])
print("arrival_spread_ns:", matched_config["sipm_response"]["arrival_spread_ns"])


## 7. Simulate feature pool with saved seeds


In [ ]:
def simulate_feature_pool_with_seeds(photon_events, config, max_events=20000, random_seed=42, threshold_mV=18.0, n_pretrigger_samples=3):
    n_events = min(max_events, len(photon_events))
    selected_events = photon_events.sample(n=n_events, random_state=random_seed).reset_index(drop=True)
    rows = []
    for local_index, (_, event_row) in enumerate(selected_events.iterrows()):
        simulation_seed = int(random_seed * 1_000_000 + local_index)
        rng = np.random.default_rng(simulation_seed)
        feature_row = simulate_feature_row_from_photon_event(
            event_row=event_row,
            config=config,
            threshold_mV=threshold_mV,
            n_pretrigger_samples=n_pretrigger_samples,
            rng=rng,
        )
        feature_row["simulation_seed"] = simulation_seed
        feature_row["pool_local_index"] = local_index
        rows.append(feature_row)
    return pd.DataFrame(rows)

sim_pool_features = simulate_feature_pool_with_seeds(
    photon_events=corsika_events_all,
    config=matched_config,
    max_events=MAX_POOL_EVENTS,
    random_seed=RANDOM_SEED,
    threshold_mV=FEATURE_THRESHOLD_MV,
    n_pretrigger_samples=N_PRETRIGGER_SAMPLES,
)
print("Simulated pool rows:", len(sim_pool_features))
print("Valid by feature threshold:", int(sim_pool_features["valid"].sum()))
print("Invalid by feature threshold:", int((~sim_pool_features["valid"]).sum()))
display(sim_pool_features[["event_id", "detector", "generated_photons", "raw_peak_mV", "peak_mV", "valid", "simulation_seed"]].head())
display(sim_pool_features[["raw_peak_mV", "peak_mV", "width_ns", "integral_mVns"]].describe())
sim_pool_features.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_sim_pool_features_with_seeds.csv", index=False)


## 8. One-to-one peak-prioritized matching


In [ ]:
def greedy_peak_match_without_replacement(real_df, sim_df, peak_column="match_peak_mV"):
    real_sorted = real_df.reset_index(names="real_original_index").sort_values(peak_column).reset_index(drop=True)
    sim_available = sim_df.reset_index(names="sim_original_index").copy().reset_index(drop=True)
    available_positions = list(range(len(sim_available)))
    match_rows = []
    selected_sim_positions = []
    for match_id, real_row in real_sorted.iterrows():
        real_peak = float(real_row[peak_column])
        candidate_peaks = sim_available.iloc[available_positions][peak_column].to_numpy(dtype=float)
        best_local_position = int(np.argmin(np.abs(candidate_peaks - real_peak)))
        best_global_position = available_positions.pop(best_local_position)
        sim_row = sim_available.iloc[best_global_position]
        sim_peak = float(sim_row[peak_column])
        selected_sim_positions.append(best_global_position)
        match_rows.append({
            "match_id": int(match_id),
            "real_original_index": int(real_row["real_original_index"]),
            "sim_original_index": int(sim_row["sim_original_index"]),
            "real_event_id": str(real_row["event_id"]),
            "simulated_event_id": str(sim_row["event_id"]),
            "real_match_peak_mV": real_peak,
            "simulated_match_peak_mV": sim_peak,
            "delta_peak_mV": sim_peak - real_peak,
            "abs_delta_peak_mV": abs(sim_peak - real_peak),
        })
    match_table = pd.DataFrame(match_rows)
    matched_real = real_sorted.copy()
    matched_sim = sim_available.iloc[selected_sim_positions].copy().reset_index(drop=True)
    matched_real["match_id"] = match_table["match_id"].to_numpy(dtype=int)
    matched_sim["match_id"] = match_table["match_id"].to_numpy(dtype=int)
    matched_sim["matched_real_event_id"] = match_table["real_event_id"].to_numpy()
    matched_sim["abs_delta_peak_mV"] = match_table["abs_delta_peak_mV"].to_numpy(dtype=float)
    return matched_real, matched_sim, match_table

real_match = real_features.copy().reset_index(drop=True)
real_match["match_peak_mV"] = pd.to_numeric(real_match["raw_peak_mV"], errors="coerce")
sim_match_pool = sim_pool_features.copy().reset_index(drop=True)
sim_match_pool["match_peak_mV"] = pd.to_numeric(sim_match_pool["raw_peak_mV"], errors="coerce")
real_match = real_match[np.isfinite(real_match["match_peak_mV"])].copy().reset_index(drop=True)
sim_match_pool = sim_match_pool[np.isfinite(sim_match_pool["match_peak_mV"])].copy().reset_index(drop=True)

print("Real HDF5 pulses available for matching:", len(real_match))
print("Simulated candidates available for matching:", len(sim_match_pool))
if len(sim_match_pool) < len(real_match):
    raise ValueError(
        f"Not enough simulated candidates for one-to-one matching: {len(sim_match_pool)} simulated vs {len(real_match)} real. "
        "Increase MAX_POOL_EVENTS or reduce an explicitly selected real subset."
    )

matched_real, matched_sim, peak_match_table = greedy_peak_match_without_replacement(real_match, sim_match_pool)
print("Matched pairs:", len(peak_match_table))
display(peak_match_table.describe())
display(peak_match_table.head())

matched_real.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_matched_real_by_raw_peak.csv", index=False)
matched_sim.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_matched_sim_by_raw_peak.csv", index=False)
peak_match_table.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_matching_summary.csv", index=False)


## 9. Normalized histogram helpers


In [ ]:
TEMPORAL_BIN_NS = 8.0
print("Red Pitaya FPGA temporal resolution is fixed at 8 ns; all temporal histograms use 8 ns bins.")
print("Estimated sampling interval [ns] used only as consistency check:", real_sampling_interval_ns)
if abs(real_sampling_interval_ns - TEMPORAL_BIN_NS) > 0.25:
    print("WARNING: estimated sampling interval differs from 8 ns by more than 0.25 ns; check timing columns before interpreting temporal histograms.")


def temporal_bins_8ns(*arrays, margin_bins=1):
    values = np.concatenate([np.asarray(array, dtype=float) for array in arrays])
    values = values[np.isfinite(values)]
    if len(values) == 0:
        raise ValueError("No finite values for temporal bins.")
    lower = TEMPORAL_BIN_NS * np.floor(values.min() / TEMPORAL_BIN_NS) - margin_bins * TEMPORAL_BIN_NS
    upper = TEMPORAL_BIN_NS * np.ceil(values.max() / TEMPORAL_BIN_NS) + margin_bins * TEMPORAL_BIN_NS
    return np.arange(lower, upper + TEMPORAL_BIN_NS, TEMPORAL_BIN_NS)


def common_linear_bins(*arrays, n_bins=40):
    values = np.concatenate([np.asarray(array, dtype=float) for array in arrays])
    values = values[np.isfinite(values)]
    lower = values.min()
    upper = values.max()
    if lower == upper:
        lower -= 0.5
        upper += 0.5
    return np.linspace(lower, upper, n_bins + 1)

def normalized_weights(values):
    values = np.asarray(values)
    return np.ones(len(values), dtype=float) / max(len(values), 1)

def plot_normalized_hist_comparison(reference, candidate, bins, xlabel, title, reference_label, candidate_label, output_path):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(reference, bins=bins, weights=normalized_weights(reference), histtype="step", linewidth=1.8, label=reference_label)
    ax.hist(candidate, bins=bins, weights=normalized_weights(candidate), histtype="step", linewidth=1.8, label=candidate_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Fraction")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    save_figure(fig, output_path)
    plt.show()

def plot_side_by_side_hist2d_normalized(real_df, sim_df, x_column, y_column, x_label, y_label, title_left, title_right, output_path, x_bins=40, y_bins=40):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
    for ax, df, title in [(axes[0], real_df, title_left), (axes[1], sim_df, title_right)]:
        weights = np.ones(len(df), dtype=float) / max(len(df), 1)
        hist = ax.hist2d(df[x_column], df[y_column], bins=[x_bins, y_bins], weights=weights)
        fig.colorbar(hist[3], ax=ax, label="Fraction")
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(title)
    fig.tight_layout()
    save_figure(fig, output_path)
    plt.show()


## 10. Normalized 1D and 2D histograms


In [ ]:
plot_normalized_hist_comparison(
    reference=matched_real["match_peak_mV"],
    candidate=matched_sim["match_peak_mV"],
    bins=common_linear_bins(matched_real["match_peak_mV"], matched_sim["match_peak_mV"], n_bins=40),
    xlabel="Raw peak [mV]",
    title=f"Normalized raw peak: latest HDF5 panel {PRIMARY_PANEL} vs peak-matched simulation",
    reference_label=f"Latest HDF5 panel {PRIMARY_PANEL}",
    candidate_label="Peak-matched simulated pulses",
    output_path=OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_hist_peak_normalized.png",
)

for feature in ["rms_mV", "integral_mVns", "width_ns"]:
    real_values = pd.to_numeric(matched_real[feature], errors="coerce")
    sim_values = pd.to_numeric(matched_sim[feature], errors="coerce")
    mask = np.isfinite(real_values) & np.isfinite(sim_values)
    if feature == "width_ns":
        bins = temporal_bins_8ns(real_values[mask], sim_values[mask])
        xlabel = "Width [ns]"
    else:
        bins = common_linear_bins(real_values[mask], sim_values[mask], n_bins=40)
        xlabel = feature
    plot_normalized_hist_comparison(
        reference=real_values[mask],
        candidate=sim_values[mask],
        bins=bins,
        xlabel=xlabel,
        title=f"Normalized {feature}: latest HDF5 panel {PRIMARY_PANEL} peak matching",
        reference_label=f"HDF5 panel {PRIMARY_PANEL}",
        candidate_label="Peak-matched simulation",
        output_path=OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_hist_{feature}_normalized.png",
    )

mask_integral = (
    np.isfinite(pd.to_numeric(matched_real["integral_mVns"], errors="coerce"))
    & np.isfinite(pd.to_numeric(matched_sim["integral_mVns"], errors="coerce"))
)
real_int_peak = matched_real[mask_integral].copy()
sim_int_peak = matched_sim[mask_integral].copy()
plot_side_by_side_hist2d_normalized(
    real_df=real_int_peak,
    sim_df=sim_int_peak,
    x_column="match_peak_mV",
    y_column="integral_mVns",
    x_label="Raw peak [mV]",
    y_label="Integral [mV ns]",
    title_left=f"HDF5 panel {PRIMARY_PANEL}",
    title_right="Peak-matched simulation",
    output_path=OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_hist2d_integral_vs_raw_peak_normalized.png",
    x_bins=common_linear_bins(real_int_peak["match_peak_mV"], sim_int_peak["match_peak_mV"], n_bins=40),
    y_bins=common_linear_bins(real_int_peak["integral_mVns"], sim_int_peak["integral_mVns"], n_bins=40),
)

mask_width_integral = (
    np.isfinite(pd.to_numeric(matched_real["width_ns"], errors="coerce"))
    & np.isfinite(pd.to_numeric(matched_sim["width_ns"], errors="coerce"))
    & np.isfinite(pd.to_numeric(matched_real["integral_mVns"], errors="coerce"))
    & np.isfinite(pd.to_numeric(matched_sim["integral_mVns"], errors="coerce"))
)
real_width_int = matched_real[mask_width_integral].copy()
sim_width_int = matched_sim[mask_width_integral].copy()
plot_side_by_side_hist2d_normalized(
    real_df=real_width_int,
    sim_df=sim_width_int,
    x_column="width_ns",
    y_column="integral_mVns",
    x_label="Width [ns]",
    y_label="Integral [mV ns]",
    title_left=f"HDF5 panel {PRIMARY_PANEL}",
    title_right="Peak-matched simulation",
    output_path=OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_hist2d_integral_vs_width_normalized.png",
    x_bins=temporal_bins_8ns(real_width_int["width_ns"], sim_width_int["width_ns"]),
    y_bins=common_linear_bins(real_width_int["integral_mVns"], sim_width_int["integral_mVns"], n_bins=40),
)


## 11. Waveform overlays


In [ ]:
def reconstruct_simulated_waveform(row, config):
    rng = np.random.default_rng(int(row["simulation_seed"]))
    result = simulate_detector_event_from_config(
        generated_photons=int(row["generated_photons"]),
        config=config,
        rng=rng,
    )
    return result.sampled_time_ns, result.digitized_voltage_mV

sim_matched_waveforms = {
    str(row["event_id"]): reconstruct_simulated_waveform(row, matched_config)
    for _, row in matched_sim.iterrows()
}

all_real_waveforms = [real_waveforms[event_id] for event_id in matched_real["event_id"] if event_id in real_waveforms]
all_sim_waveforms = [sim_matched_waveforms[event_id] for event_id in matched_sim["event_id"] if event_id in sim_matched_waveforms]

fig, ax = plt.subplots(figsize=(7, 4))
plot_waveform_overlay(all_real_waveforms, xlabel="Time [ns]", ylabel="Voltage [mV]", label=f"All real HDF5 panel {PRIMARY_PANEL} pulses", alpha=0.04, ax=ax)
ax.set_title(f"All real HDF5 pulses: panel {PRIMARY_PANEL}")
save_figure(fig, OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_overlay_all_real_pulses.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
plot_waveform_overlay(all_sim_waveforms, xlabel="Time [ns]", ylabel="Voltage [mV]", label="All peak-matched simulated pulses", alpha=0.04, ax=ax)
ax.set_title(f"All peak-matched simulated pulses for panel {PRIMARY_PANEL}")
save_figure(fig, OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_overlay_all_matched_simulated_pulses.png")
plt.show()


In [ ]:
pair_order = peak_match_table.sort_values("real_match_peak_mV").reset_index(drop=True)
representative_positions = np.linspace(0, len(pair_order) - 1, min(N_REPRESENTATIVE_OVERLAY, len(pair_order))).astype(int)
representative_match_ids = pair_order.iloc[representative_positions]["match_id"].astype(int).tolist()

rep_real = matched_real[matched_real["match_id"].isin(representative_match_ids)].copy().sort_values("match_peak_mV")
rep_sim = matched_sim.set_index("match_id").loc[rep_real["match_id"]].reset_index()
rep_real_waveforms = [real_waveforms[event_id] for event_id in rep_real["event_id"] if event_id in real_waveforms]
rep_sim_waveforms = [sim_matched_waveforms[event_id] for event_id in rep_sim["event_id"] if event_id in sim_matched_waveforms]

fig, ax = plt.subplots(figsize=(7, 4))
plot_waveform_overlay(rep_real_waveforms, xlabel="Time [ns]", ylabel="Voltage [mV]", label=f"Representative HDF5 panel {PRIMARY_PANEL} pulses", alpha=0.25, ax=ax)
ax.set_title(f"Representative real HDF5 pulses: panel {PRIMARY_PANEL}")
save_figure(fig, OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_overlay_representative_real_pulses.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
plot_waveform_overlay(rep_sim_waveforms, xlabel="Time [ns]", ylabel="Voltage [mV]", label="Representative simulated pulses", alpha=0.25, ax=ax)
ax.set_title(f"Representative simulated pulses for panel {PRIMARY_PANEL}")
save_figure(fig, OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_overlay_representative_simulated_pulses.png")
plt.show()

pair_positions = np.linspace(0, len(pair_order) - 1, min(N_REPRESENTATIVE_PAIR_PLOTS, len(pair_order))).astype(int)
pair_ids = pair_order.iloc[pair_positions]["match_id"].astype(int).tolist()
n_cols = 3
n_rows = int(np.ceil(len(pair_ids) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 3.0 * n_rows), squeeze=False)
for ax, match_id in zip(axes.ravel(), pair_ids):
    real_row = matched_real.loc[matched_real["match_id"] == match_id].iloc[0]
    sim_row = matched_sim.loc[matched_sim["match_id"] == match_id].iloc[0]
    real_waveform = real_waveforms[str(real_row["event_id"])]
    sim_waveform = sim_matched_waveforms[str(sim_row["event_id"])]
    ax.plot(real_waveform[0], real_waveform[1], label="real", linewidth=1.2)
    ax.plot(sim_waveform[0], sim_waveform[1], label="sim", linewidth=1.2)
    ax.set_title(f"match {match_id}: peak {real_row['match_peak_mV']:.1f} vs {sim_row['match_peak_mV']:.1f} mV")
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel("Voltage [mV]")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
for ax in axes.ravel()[len(pair_ids):]:
    ax.axis("off")
fig.tight_layout()
save_figure(fig, OUTPUT_FIGURES / f"latest_h5_panel_{PRIMARY_PANEL}_overlay_representative_real_sim_pairs.png")
plt.show()


## 12. Quantitative summary


In [ ]:
comparison = compare_real_sim_features(
    matched_real,
    matched_sim,
    feature_columns=["match_peak_mV", "rms_mV", "integral_mVns", "width_ns"],
    relationships=[("match_peak_mV", "integral_mVns"), ("width_ns", "integral_mVns")],
)
comparison.distribution_comparison.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_feature_distribution_comparison.csv", index=False)
comparison.correlation_comparison.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_feature_correlation_comparison.csv", index=False)
comparison.validity_summary.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_feature_validity_summary.csv", index=False)
display(comparison.distribution_comparison)
display(comparison.correlation_comparison)
display(comparison.validity_summary)

summary = pd.DataFrame(
    {
        "metric": [
            "primary_panel",
            "segmentation_method",
            "representative_real_pulse_target",
            "n_representative_real_hdf5_pulses_used",
            "n_simulated_pool_events",
            "n_matched_pairs",
            "estimated_sampling_interval_ns",
            "temporal_bin_ns",
            "mean_abs_delta_raw_peak_mV",
            "median_abs_delta_raw_peak_mV",
            "max_abs_delta_raw_peak_mV",
            "eta_transport",
            "tau_f_ns",
            "arrival_spread_ns",
            "fixed_voltage_scale_mV_per_pe",
            "fixed_pde",
            "exploratory_all_corsika_detectors",
        ],
        "value": [
            PRIMARY_PANEL,
            SEGMENTATION_METHOD,
            REPRESENTATIVE_REAL_PULSES,
            len(real_match),
            len(sim_pool_features),
            len(peak_match_table),
            real_sampling_interval_ns,
            TEMPORAL_BIN_NS,
            peak_match_table["abs_delta_peak_mV"].mean(),
            peak_match_table["abs_delta_peak_mV"].median(),
            peak_match_table["abs_delta_peak_mV"].max(),
            ETA_TRANSPORT,
            TAU_F_NS,
            ARRIVAL_SPREAD_NS,
            matched_config["voltage_conversion"]["voltage_scale_mV_per_pe"],
            matched_config["photon_transport"]["pde"],
            True,
        ],
    }
)
display(summary)
summary.to_csv(OUTPUT_TABLES / f"latest_h5_panel_{PRIMARY_PANEL}_analysis_summary.csv", index=False)
print("Output tables:", OUTPUT_TABLES)
print("Output figures:", OUTPUT_FIGURES)
